
# **Layer 2 – Staging Layer (Clean & Typed Data)**

**Purpose:**

* Convert raw CSV strings into proper data types.
* Clean and structure data for analytics.
* Add derived fields and load timestamps.
* Basic data validation to catch bad records.

**Steps We Did:**

1. **Create Staging Tables**

   * `staging.customers` → customer data

     * Convert `customer_id` to NUMBER
     * Parse `created_date` to DATE
     * Add `load_ts`
   * `staging.products` → product data

     * Convert `product_id` and `unit_price` to NUMBER
     * Add `load_ts`
   * `staging.transactions` → transaction data

     * Convert `transaction_id`, `customer_id`, `product_id`, `quantity`, `unit_price` to NUMBER
     * Parse `transaction_date` to DATE
     * Calculate `total_amount = quantity × unit_price`
     * Add `load_ts`

2. **Handle Bad Data Safely**

   * Used `TRY_TO_NUMBER` and `TRY_TO_DATE` to avoid load failures
   * Nulls or invalid formats do not break the pipeline

3. **Basic Validation (Optional Stored Procedure)**

   * Check missing customers or products in transactions
   * Can be reused for any staging table

4. **Quick Checks**

   * Row counts per table
   * Preview a few records
   * Check for negative totals

**Key Points:**

* Data is now **typed, clean, and safe** for analytics.
* Derived metrics like `total_amount` are calculated here.
* Errors in raw data are handled safely without stopping the load.
* This layer **feeds production/fact tables** reliably.



## **1️ Purpose of the Procedure / Query**

The **goal** is to ensure **data quality in the staging layer** before we move it into facts and dimensions.

* Staging tables are **typed and clean**, but errors can still exist in CSV or raw data.
* Typical errors include:

  * Transactions referencing non-existent customers or products
  * Negative quantities or prices
  * Missing dates

This procedure / query **catches these issues early**, so downstream modeling and analytics won’t break.

---

## **2️ What Each Check Does**

| Check Name                 | What It Does                                                                  | Why It Matters                                         |
| -------------------------- | ----------------------------------------------------------------------------- | ------------------------------------------------------ |
| `missing_customers`        | Counts transactions where `customer_id` does not exist in `staging.customers` | Prevents revenue or LTV calculations from being wrong  |
| `missing_products`         | Counts transactions where `product_id` does not exist in `staging.products`   | Prevents incorrect product performance analysis        |
| `negative_quantity`        | Counts transactions where quantity < 0                                        | Data entry error / impossible scenario                 |
| `negative_unit_price`      | Counts transactions where unit price < 0                                      | Protects revenue and sales metrics from being negative |
| `missing_transaction_date` | Counts transactions where the transaction date is NULL                        | Ensures time-series analysis can be done correctly     |

---

## **3️ How the Procedure Works**

### **Option 1: JSON Procedure (what you’re running)**

1. Create a **JavaScript procedure** in Snowflake
2. Inside the procedure, we execute **multiple SQL statements** — each counting one type of data issue
3. Store each result in a **JavaScript object**
4. Convert the object to a **JSON string** and return it

**Result Example:**

```json
{
  "missing_customers": 0,
  "missing_products": 0,
  "negative_quantity": 2,
  "negative_unit_price": 0,
  "missing_transaction_date": 1
}
```

:: You get **all checks in a single call**, easy to parse in code, dashboards, or logs.

---

### **Option 2: SQL Query (union all)**

* Runs **five separate counts** in one query using `UNION ALL`
* Returns a **table with two columns**: `check_name` and `missing_count`
* Directly readable in Snowflake worksheet or BI tools

**Result Example:**

| check_name               | missing_count |
| ------------------------ | ------------- |
| missing_customers        | 0             |
| missing_products         | 0             |
| negative_quantity        | 2             |
| negative_unit_price      | 0             |
| missing_transaction_date | 1             |

:: Easy to query, visualize, or join into dashboards.

---

## **4️ Why This Is Important in a Pipeline**

1. **Catches errors early** before they propagate to facts/dimensions
2. **Ensures analytics are trustworthy**
3. **Supports automated monitoring** (you can call the procedure daily or hourly)
4. **Helps in auditing**: you can track trends in data quality over time
5. **Makes the staging layer production-ready** — ready for BI tools, dashboards, or reporting

---

### **5️ Interview-Friendly Explanation**

> “We created a staging validation procedure that scans the staging tables for common data quality issues: missing customer or product references, negative quantities or unit prices, and missing transaction dates. The procedure aggregates these checks into a single JSON output, so we can easily monitor and enforce data quality before the data is moved into fact and dimension tables. This ensures downstream analytics are accurate and reliable.”

---
